In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

# 1: define function to cap extreme outliers using the interquartile range (IQR) method
def handle_outliers_iqr(df, columns):
    df_cleaned = df.copy()
    for col in columns:
        Q1 = df_cleaned[col].quantile(0.25)
        Q3 = df_cleaned[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # cap the values (physically implausible spikes)
        df_cleaned[col] = np.where(df_cleaned[col] > upper_bound, upper_bound, df_cleaned[col])
        df_cleaned[col] = np.where(df_cleaned[col] < lower_bound, lower_bound, df_cleaned[col])
    return df_cleaned

# 2: define master function for data cleaning, chronological partitioning, and robust scaling
def process_and_split_plant(filename, plant_name):
    
    # check if Phase 1A file exists
    if not os.path.exists(filename):
        print(f"[ERROR] Cannot find {filename}. Make sure it is in the same folder.")
        return
    
    # load data
    df = pd.read_csv(filename, index_col='Time', parse_dates=True)
    features = df.columns.tolist()
    
# 3: apply outlier capping to physically implausible spikes
    df = handle_outliers_iqr(df, features)
    
# 4: partition data chronologically (60% Train, 10% Val, 15% Calib, 15% Test)
    n = len(df)
    train_end = int(n * 0.60)
    val_end = int(n * 0.70)   # 60 + 10
    calib_end = int(n * 0.85) # 70 + 15
    
    print(f"  -> Splitting {n} rows chronologically...")
    df_train = df.iloc[:train_end].copy()
    df_val = df.iloc[train_end:val_end].copy()
    df_calib = df.iloc[val_end:calib_end].copy()
    df_test = df.iloc[calib_end:].copy()
    
# 5: apply robust scaling strictly fitted on training data to prevent data leakage
    scaler = RobustScaler()
    scaler.fit(df_train) # fit to training data only to avoid data leakage
    
    df_train[features] = scaler.transform(df_train)
    df_val[features] = scaler.transform(df_val)
    df_calib[features] = scaler.transform(df_calib)
    df_test[features] = scaler.transform(df_test)
    
# 6: export partitioned datasets to designated directories
    output_dir = f'Model_Ready_Data_{plant_name}'
    os.makedirs(output_dir, exist_ok=True)
    
    df_train.to_csv(os.path.join(output_dir, 'Train.csv'))
    df_val.to_csv(os.path.join(output_dir, 'Validation.csv'))
    df_calib.to_csv(os.path.join(output_dir, 'Calibration.csv'))
    df_test.to_csv(os.path.join(output_dir, 'Test.csv'))
    
# 7: execute data pipeline for Agus I and II
process_and_split_plant('Processed_Agus1_Dataset.csv', 'Agus1')
process_and_split_plant('Processed_Agus2_Dataset.csv', 'Agus2')

Starting Data Cleaning, Partitioning, and Robust Scaling...

--- Processing Agus1 ---
  -> Capping outliers using IQR method...
  -> Splitting 52513 rows chronologically...
  -> Applying Robust Scaling (Median/IQR)...
  [SUCCESS] Agus1 data saved in folder: 'Model_Ready_Data_Agus1'
      Train: 31507 rows
      Val:   5252 rows
      Calib: 7877 rows
      Test:  7877 rows

--- Processing Agus2 ---
  -> Capping outliers using IQR method...
  -> Splitting 52513 rows chronologically...
  -> Applying Robust Scaling (Median/IQR)...
  [SUCCESS] Agus2 data saved in folder: 'Model_Ready_Data_Agus2'
      Train: 31507 rows
      Val:   5252 rows
      Calib: 7877 rows
      Test:  7877 rows

*** PHASE 1 OFFICIALLY COMPLETE! ***
